In [1]:
pip install pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import os

folder_path = r"C:\Users\user\Documents\Senior Thesis Excel Files\Excel Files"

file_names = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

print(file_names)

['Combined Jeonse and Monthly Rent Index.xlsx', 'Integrated Monthly Rent Price Index by Property Type.xlsx', 'Monthly Rent Supply and Demand Trends by Major Region.xlsx', 'Property Price Index by Type.xlsx', 'Ratio of Jeonse (rental deposit) price to sales price by property type.xlsx', 'Ratio of Monthly Rent Deposit to Jeonse Price by Property Type.xlsx']


In [3]:
df_dict = {}
df_arr = []
for file in file_names:
    file_name = os.path.join(folder_path,file)
    df_arr.append(pd.read_excel(file_name, sheet_name="데이터"))
print(df_arr)

[        Date  Comprehensive Housing Index  Apartment  \
0    2015.06                         91.3       88.0   
1    2015.07                         91.7       88.5   
2    2015.08                         92.0       88.9   
3    2015.09                         92.4       89.6   
4    2015.10                         92.9       90.2   
..       ...                          ...        ...   
113  2024.11                         97.1       95.4   
114  2024.12                         97.1       95.5   
115  2025.01                         97.2       95.5   
116  2025.02                         97.2       95.6   
117  2025.03                         97.4       95.8   

     Low-Rise Multi-Unit Housing  Single-Family House  
0                           94.6                 97.7  
1                           94.7                 97.8  
2                           94.9                 97.8  
3                           95.2                 97.9  
4                           95.4              

In [4]:
col_names = df_arr[0].copy().columns
print(col_names)
print(file_names)

for i in range(0,len(df_arr)):
    temp = df_arr[i]
    temp = temp.rename(columns = {col: f"{file_names[i]}_{col}" for col in temp.columns if col != "Date"})
    df_arr[i] = temp
print(df_arr[0].columns)

#print(df_arr[1])

Index(['Date', 'Comprehensive Housing Index', 'Apartment',
       'Low-Rise Multi-Unit Housing', 'Single-Family House'],
      dtype='object')
['Combined Jeonse and Monthly Rent Index.xlsx', 'Integrated Monthly Rent Price Index by Property Type.xlsx', 'Monthly Rent Supply and Demand Trends by Major Region.xlsx', 'Property Price Index by Type.xlsx', 'Ratio of Jeonse (rental deposit) price to sales price by property type.xlsx', 'Ratio of Monthly Rent Deposit to Jeonse Price by Property Type.xlsx']
Index(['Date',
       'Combined Jeonse and Monthly Rent Index.xlsx_Comprehensive Housing Index',
       'Combined Jeonse and Monthly Rent Index.xlsx_Apartment',
       'Combined Jeonse and Monthly Rent Index.xlsx_Low-Rise Multi-Unit Housing',
       'Combined Jeonse and Monthly Rent Index.xlsx_Single-Family House'],
      dtype='object')


In [5]:
from functools import reduce

merged = reduce(lambda left, right: pd.merge(left,right,on="Date",how="outer"),df_arr)
print(merged)

        Date  \
0    2003.11   
1    2003.12   
2    2004.01   
3    2004.02   
4    2004.03   
..       ...   
254  2025.01   
255  2025.02   
256  2025.03   
257  2025.04   
258  2025.05   

     Combined Jeonse and Monthly Rent Index.xlsx_Comprehensive Housing Index  \
0                                                  NaN                         
1                                                  NaN                         
2                                                  NaN                         
3                                                  NaN                         
4                                                  NaN                         
..                                                 ...                         
254                                               97.2                         
255                                               97.2                         
256                                               97.4                         
257    

In [6]:
import numpy as np
new_df_arr = []
for i in range(1,len(col_names)):
    phrase = col_names[i]
    cols = [col for col in merged.columns if phrase in col]
    cols.insert(0,"Date")
    new_df_arr.append(merged[cols].copy())

# print(len(new_df_arr))

column_names = ["Date",
                "Combined Jeonse and Monthly Rent Index",
                "Integrated Monthly Rent Price Index",
                "Monthly Rent Supply and Demand Trends",
                "Property Price Index",
                "Price/Jeonse",
                "Ratio of Monthly Rent Deposit to Jeonse Price"]

tightening_dates = ["2017.08","2018.09","2019.12"]
loosening_dates = ["2022.12"]
tightening_dates = pd.to_datetime(tightening_dates, format="%Y.%m")
loosening_dates  = pd.to_datetime(loosening_dates, format="%Y.%m")

for df in new_df_arr:
    df.columns = column_names
    df.dropna(inplace=True)

    # Creating the new ratios to be tested in OLS/log transforming for better testing to recenter around 1
    df["Property Price Index"] = df["Property Price Index"]/df["Property Price Index"].iloc[0]
    df["Price/Nonownership Ratio"] = np.log((df["Property Price Index"])/df["Combined Jeonse and Monthly Rent Index"])
    df["Price/Rent"] = np.log((df["Property Price Index"])/df["Integrated Monthly Rent Price Index"])
    df["Price/Jeonse"] = np.log(100/df["Price/Jeonse"])
    
    # Standardizing Time to be Parsable as an index
    df["Date"] = pd.to_datetime(df["Date"].astype(str), format="%Y.%m")
    df["Policy Dummy"] = 0
    df.loc[df["Date"].isin(tightening_dates), "Policy Dummy"] = 1
    df.loc[df["Date"].isin(loosening_dates),  "Policy Dummy"] = -1
    df = df.set_index("Date")



In [7]:
with pd.ExcelWriter("output3.xlsx") as writer:
    for i, df in enumerate(new_df_arr):
        sheet_name = f"{col_names[i+1]}"
        df.to_excel(writer, sheet_name=sheet_name,index=False)